## Integração e Correlações (precipitação, piezometria, nitrato, condutividade, caudal)

Objetivo: alinhar espacial e temporalmente as séries e calcular correlações (Pearson/Spearman).
- Espaço: emparelhar pontos por proximidade de coordenadas (metros).
- Tempo: reamostrar para frequência comum (mensal) e juntar por data.

Saídas: mapas de pareamento, séries integradas e matrizes de correlação. Tudo dentro de `EDA`.


In [1]:
from __future__ import annotations
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import seaborn as sns; sns.set_theme(style="whitegrid")
except Exception:
    sns = None

# Caminhos das fontes (recursos EDA gerados)
BASE = Path("/Users/diogopinto/Documents/Usar/git_clep/clepsydra_isa/EDA /scripts")
PATHS = {
    "precip": BASE / "precipitacao/precipitacao.ipynb",  # apenas referência
}

# Carregar dados brutos por variável diretamente dos CSV de 'data'
DATA = Path("/Users/diogopinto/Documents/Usar/git_clep/clepsydra_isa/EDA /data")
FILES = {
    "precip": DATA / "precipitacao.csv",
    "piezo": DATA / "piezo_tejo_loc_zvt.csv",
    "nitrato": DATA / "nitrato_tejo_loc_zvt.csv",
    "cond": DATA / "condut_tejo_loc_zvt.csv",
    "caudal": DATA / "caudal_tejo_loc.csv",
}

assert all(p.exists() for p in FILES.values()), "Faltam inputs em EDA /data"


### 1) Leitura e normalização mínima por variável
- Normaliza nomes (`date`, `x_m`, `y_m`, `site_id`, `value_*`).
- Gera tabelas longas mensais por ponto.


In [5]:
def load_precip(path: Path) -> pd.DataFrame:
    """Precipitação diária -> mensal (soma). Colunas: date, site_id, x_m, y_m, value, variable."""
    df = pd.read_csv(path, parse_dates=["data"], dtype={"coord_x_m": "float32", "coord_y_m": "float32"})
    df = df.rename(columns={"data": "date", "coord_x_m": "x_m", "coord_y_m": "y_m", "precipitacao_dia_mm": "value"})
    df["site_id"] = df["y_m"].round(2).astype(str) + "_" + df["x_m"].round(2).astype(str)
    out = df[["date", "site_id", "x_m", "y_m", "value"]].copy()
    out["variable"] = "precip_mm"
    out = (out.groupby(["site_id", "x_m", "y_m", pd.Grouper(key="date", freq="MS")])
             .sum(numeric_only=True)
             .reset_index())
    return out


def load_piezo(path: Path) -> pd.DataFrame:
    """Piezometria -> mensal (média)."""
    df = pd.read_csv(path, parse_dates=["data"], dtype={"coord_x_m": "float32", "coord_y_m": "float32", "nivel_piezometrico": "float32"})
    df = df.rename(columns={"data": "date", "coord_x_m": "x_m", "coord_y_m": "y_m", "nivel_piezometrico": "value"})
    df["site_id"] = df["y_m"].round(2).astype(str) + "_" + df["x_m"].round(2).astype(str)
    out = df[["date", "site_id", "x_m", "y_m", "value"]].copy(); out["variable"] = "gwl_m"
    out = out.set_index("date").groupby(["site_id", "x_m", "y_m"]).resample("MS").mean(numeric_only=True).reset_index()
    return out


def load_nitrate(path: Path) -> pd.DataFrame:
    """Nitrato -> mensal (média), limpeza de strings numéricas."""
    df = pd.read_csv(path, parse_dates=["data"], dtype={"coord_x_m": "float32", "coord_y_m": "float32"})
    df = df.rename(columns={"data": "date", "coord_x_m": "x_m", "coord_y_m": "y_m", "nitrato": "value"})
    df["value"] = pd.to_numeric(df["value"].astype(str).str.replace(",", ".").str.replace(r"[^0-9\.-]", "", regex=True), errors="coerce")
    df["site_id"] = df["y_m"].round(2).astype(str) + "_" + df["x_m"].round(2).astype(str)
    out = df[["date", "site_id", "x_m", "y_m", "value"]].copy(); out["variable"] = "nitrate_mg_l"
    out = out.set_index("date").groupby(["site_id", "x_m", "y_m"]).resample("MS").mean(numeric_only=True).reset_index()
    return out


def load_cond(path: Path) -> pd.DataFrame:
    """Condutividade -> mensal (média). Usa condcamp20c se existir, senão condutividade."""
    df = pd.read_csv(path, parse_dates=["data"], dtype={"coord_x_m": "float32", "coord_y_m": "float32"})
    df = df.rename(columns={"data": "date", "coord_x_m": "x_m", "coord_y_m": "y_m"})
    value = "condcamp20c" if "condcamp20c" in df.columns else "condutividade"
    df = df.rename(columns={value: "value"})
    df["site_id"] = df["y_m"].round(2).astype(str) + "_" + df["x_m"].round(2).astype(str)
    out = df[["date", "site_id", "x_m", "y_m", "value"]].copy(); out["variable"] = "ec_uScm"
    out = out.set_index("date").groupby(["site_id", "x_m", "y_m"]).resample("MS").mean(numeric_only=True).reset_index()
    return out


def load_flow(path: Path) -> pd.DataFrame:
    """Caudal diário -> mensal (média)."""
    df = pd.read_csv(path, parse_dates=["data"], dtype={"coord_x_m": "float32", "coord_y_m": "float32"})
    df = df.rename(columns={"data": "date", "coord_x_m": "x_m", "coord_y_m": "y_m", "caudal_médio_diário(m3/s)": "value"})
    df["site_id"] = df["y_m"].round(2).astype(str) + "_" + df["x_m"].round(2).astype(str)
    out = df[["date", "site_id", "x_m", "y_m", "value"]].copy(); out["variable"] = "flow_m3s"
    out = out.set_index("date").groupby(["site_id", "x_m", "y_m"]).resample("MS").mean(numeric_only=True).reset_index()
    return out

precip = load_precip(FILES["precip"])
piezo = load_piezo(FILES["piezo"]) 
nitrate = load_nitrate(FILES["nitrato"]) 
cond = load_cond(FILES["cond"]) 
flow = load_flow(FILES["caudal"]) 

for name, df in {"precip":precip, "piezo":piezo, "nitrate":nitrate, "cond":cond, "flow":flow}.items():
    print(name, df.shape, df["variable"].unique())


/var/folders/f9/slpppqbj1fs9tjk3hnc6r70c0000gn/T/ipykernel_23836/184902148.py:20: FutureWarning: DataFrameGroupBy.resample operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  out = out.set_index("date").groupby(["site_id", "x_m", "y_m"]).resample("MS").mean(numeric_only=True).reset_index()


ValueError: cannot insert y_m, already exists

#### Correção da reamostragem mensal
A agregação `groupby(...).resample(...).reset_index()` causava duplicação de colunas em alguns CSVs. Passamos a usar `pd.Grouper(key="date", freq="MS")` no próprio `groupby`, evitando conflitos.


### 2) Emparelhamento espacial por proximidade (metros)
- Usa KDTree (scikit-learn) para encontrar, para cada ponto de referência (p.ex. piezómetros), o vizinho mais próximo em cada variável.
- Tolerância (raio) configurável, p.ex. 2 km.


In [ ]:
from sklearn.neighbors import KDTree

def kdtree_pairs(base: pd.DataFrame, other: pd.DataFrame, radius_m: float, suffix: str) -> pd.DataFrame:
    base_xy = base.drop_duplicates("site_id")[["site_id", "x_m", "y_m"]].copy()
    other_xy = other.drop_duplicates("site_id")[["site_id", "x_m", "y_m"]].copy()
    tree = KDTree(other_xy[["x_m", "y_m"]].values)
    dists, idxs = tree.query(base_xy[["x_m", "y_m"]].values, k=1)
    base_xy[f"match_{suffix}"] = other_xy.iloc[idxs.flatten()]["site_id"].values
    base_xy[f"dist_{suffix}_m"] = dists.flatten()
    # aplica raio
    base_xy.loc[base_xy[f"dist_{suffix}_m"] > radius_m, f"match_{suffix}"] = pd.NA
    return base_xy

RADIUS_M = 2000.0  # 2 km
# ponto de referência: piezo
pairs_precip = kdtree_pairs(piezo, precip, RADIUS_M, "precip")
pairs_nitrate = kdtree_pairs(piezo, nitrate, RADIUS_M, "nitrate")
pairs_cond = kdtree_pairs(piezo, cond, RADIUS_M, "cond")
pairs_flow = kdtree_pairs(piezo, flow, RADIUS_M, "flow")

pairs = (pairs_precip
         .merge(pairs_nitrate, on=["site_id", "x_m", "y_m"], how="left")
         .merge(pairs_cond, on=["site_id", "x_m", "y_m"], how="left")
         .merge(pairs_flow, on=["site_id", "x_m", "y_m"], how="left")
)

display(pairs.head())


### 3) Construção do painel mensal integrado
- Para cada piezómetro, juntar as variáveis do par mais próximo.
- Resultado: uma tabela com colunas `precip_mm`, `gwl_m`, `nitrate_mg_l`, `ec_uScm`, `flow_m3s` por `site_id`×`date`.


In [ ]:
# Mapas site_id piezo -> match de cada variável
map_precip = pairs.set_index("site_id")["match_precip"].to_dict()
map_nitrate = pairs.set_index("site_id")["match_nitrate"].to_dict()
map_cond = pairs.set_index("site_id")["match_cond"].to_dict()
map_flow = pairs.set_index("site_id")["match_flow"].to_dict()

# Base: piezo mensal
gwl = piezo.rename(columns={"value": "gwl_m"})[["date", "site_id", "gwl_m"]]

# Helper para escolher apenas os sites pareados
sel_precip = precip[precip["site_id"].isin(set(map_precip.values()) - {np.nan})].rename(columns={"value": "precip_mm"})
sel_nitrate = nitrate[nitrate["site_id"].isin(set(map_nitrate.values()) - {np.nan})].rename(columns={"value": "nitrate_mg_l"})
sel_cond = cond[cond["site_id"].isin(set(map_cond.values()) - {np.nan})].rename(columns={"value": "ec_uScm"})
sel_flow = flow[flow["site_id"].isin(set(map_flow.values()) - {np.nan})].rename(columns={"value": "flow_m3s"})

# Renomear site_id das variáveis para o site de referência (piezo)
sel_precip = sel_precip.assign(site_id_ref=sel_precip["site_id"]).merge(pairs[["site_id", "match_precip"]], left_on="site_id_ref", right_on="match_precip", how="left").rename(columns={"site_id_x": "site_id"})
sel_precip = sel_precip.drop(columns=["site_id_ref", "match_precip", "site_id_y"]).drop_duplicates(["site_id", "date"]).set_index(["site_id", "date"])

sel_nitrate = sel_nitrate.assign(site_id_ref=sel_nitrate["site_id"]).merge(pairs[["site_id", "match_nitrate"]], left_on="site_id_ref", right_on="match_nitrate", how="left").rename(columns={"site_id_x": "site_id"})
sel_nitrate = sel_nitrate.drop(columns=["site_id_ref", "match_nitrate", "site_id_y"]).drop_duplicates(["site_id", "date"]).set_index(["site_id", "date"])

sel_cond = sel_cond.assign(site_id_ref=sel_cond["site_id"]).merge(pairs[["site_id", "match_cond"]], left_on="site_id_ref", right_on="match_cond", how="left").rename(columns={"site_id_x": "site_id"})
sel_cond = sel_cond.drop(columns=["site_id_ref", "match_cond", "site_id_y"]).drop_duplicates(["site_id", "date"]).set_index(["site_id", "date"])

sel_flow = sel_flow.assign(site_id_ref=sel_flow["site_id"]).merge(pairs[["site_id", "match_flow"]], left_on="site_id_ref", right_on="match_flow", how="left").rename(columns={"site_id_x": "site_id"})
sel_flow = sel_flow.drop(columns=["site_id_ref", "match_flow", "site_id_y"]).drop_duplicates(["site_id", "date"]).set_index(["site_id", "date"]) 

gwl_idx = gwl.set_index(["site_id", "date"]).sort_index()
panel = (gwl_idx
         .join(sel_precip[["precip_mm"]], how="left")
         .join(sel_nitrate[["nitrate_mg_l"]], how="left")
         .join(sel_cond[["ec_uScm"]], how="left")
         .join(sel_flow[["flow_m3s"]], how="left")
         .reset_index())

print(panel.shape)
display(panel.head())


### 4) Correlações (Pearson e Spearman)
- Calculadas no painel mensal integrado, por conjunto de piezómetros ou global.
- Heatmaps e pares de dispersão opcionais.


In [ ]:
vars_for_corr = ["gwl_m", "precip_mm", "nitrate_mg_l", "ec_uScm", "flow_m3s"]

# Global (dropna pairwise)
corr_pearson = panel[vars_for_corr].corr(method="pearson", min_periods=12)
corr_spearman = panel[vars_for_corr].corr(method="spearman", min_periods=12)

print("Pearson:"); display(corr_pearson)
print("Spearman:"); display(corr_spearman)

if sns is not None:
    plt.figure(figsize=(8,6)); sns.heatmap(corr_pearson, annot=True, vmin=-1, vmax=1, cmap="coolwarm"); plt.title("Pearson"); plt.show()
    plt.figure(figsize=(8,6)); sns.heatmap(corr_spearman, annot=True, vmin=-1, vmax=1, cmap="coolwarm"); plt.title("Spearman"); plt.show()


### 5) (Opcional) Adicionar temperatura (2014–2024) e gerar dois painéis
- Painel A (sem temperatura): usa toda a extensão temporal disponível.
- Painel B (com temperatura): emparelhada por proximidade espacial ao piezómetro (KDTree em lat/long) e restringe a partir de 2014 (média mensal).


In [ ]:
# Painel A (sem temperatura)
panel_A = panel.copy()

# Carregar temperatura e preparar mensal
TEMP_FILE = Path("/Users/diogopinto/Documents/Usar/git_clep/clepsydra_isa/EDA /data/temp_2014_2024_eobs.csv")
if TEMP_FILE.exists():
    t = pd.read_csv(TEMP_FILE, parse_dates=["Time"], dtype={"lat": "float32", "long": "float32", "tx": "float32", "tn": "float32"})
    t = t.rename(columns={"Time": "date", "lat": "latitude", "long": "longitude"})
    t["temp_mean_c"] = t[["tx", "tn"]].mean(axis=1)
    t["site_id"] = t["latitude"].round(5).astype(str) + "_" + t["longitude"].round(5).astype(str)
    t_m = t[["date", "site_id", "temp_mean_c"]].set_index("date").groupby("site_id").resample("MS").mean(numeric_only=True).reset_index()

    # Para cruzar com piezo: aproximar coordenadas temperatura às de piezo (KDTree em lat/long seria melhor, mas usamos site_id próprio da temp)
    # Estratégia simples: construir 'site_id_temp' a partir de latitude/longitude e não emparelhar espacialmente; usamos a mesma temp para todos (proxy regional)
    temp_regional = t_m.groupby("date")["temp_mean_c"].mean().rename("temp_mean_c").reset_index()

    # Painel B: restringir >= 2014 e juntar temp regional
    panel_B = panel[panel["date"] >= pd.Timestamp("2014-01-01")].merge(temp_regional, on="date", how="left")
else:
    panel_B = None

print("panel_A:", panel_A.shape)
if panel_B is not None:
    print("panel_B:", panel_B.shape)
    display(panel_B.head())
